# Multi-Tool Party Agent

In this notebook, I build a more advanced AI agent using multiple tools with the Hugging Face `smolagents` library.

The agent has access to different capabilities, including web search, webpage access, menu suggestions, catering recommendations, and superhero-themed party ideas.

In this notebook, I will learn how to:

- Combine multiple tools into a single `CodeAgent`
- Create custom tools using the `@tool` decorator
- Create a custom tool by inheriting from the `Tool` class
- Allow an agent to decide which tool is appropriate for a task
- Use a language model as the reasoning engine of a multi-tool agent

## 1. Importing Libraries and Creating the Model

First, I import the components required to build the multi-tool agent.

The agent will use `InferenceClientModel` as its language model and will have access to built-in tools and custom tools.

In [1]:
from smolagents import (
    CodeAgent,
    DuckDuckGoSearchTool,
    FinalAnswerTool,
    InferenceClientModel,
    Tool,
    tool,
    VisitWebpageTool
)

model = InferenceClientModel(
    model_id="Qwen/Qwen2.5-Coder-32B-Instruct"
)

## 2. Creating the Menu Suggestion Tool

The first custom tool suggests a menu based on the type of occasion.

I use the `@tool` decorator to convert a Python function into a tool that can be used by the `CodeAgent`.

In [2]:
@tool
def suggest_menu(occasion: str) -> str:
    """
    Suggests a menu based on the occasion.

    Args:
        occasion (str): The type of occasion for the party.
    """
    if occasion == "casual":
        return "Pizza, snacks, and drinks."
    elif occasion == "formal":
        return "3-course dinner with wine and dessert."
    elif occasion == "superhero":
        return "Buffet with high-energy and healthy food."
    else:
        return "Custom menu for the butler."

## 3. Creating the Catering Service Tool

The second custom tool helps the agent identify the highest-rated catering service.

This example demonstrates how a Python function can be converted into an agent tool using the `@tool` decorator.

In [3]:
@tool
def catering_service_tool(query: str) -> str:
    """
    This tool returns the highest-rated catering service in Gotham City.

    Args:
        query (str): A search term for finding catering services.
    """
    services = {
        "Gotham Catering Co.": 4.9,
        "Wayne Manor Catering": 4.8,
        "Gotham City Events": 4.7,
    }

    best_service = max(services, key=services.get)

    return best_service

## 4. Creating the Superhero Party Theme Tool

In the previous examples, I created custom tools using the `@tool` decorator.

Here, I create a more structured custom tool by creating a class that inherits from the `Tool` class. The tool accepts a party category and returns a suitable superhero-themed party idea.

In [4]:
class SuperheroPartyThemeTool(Tool):
    name = "superhero_party_theme_generator"

    description = """
    This tool suggests creative superhero-themed party ideas based on a category.
    It returns a unique party theme idea.
    """

    inputs = {
        "category": {
            "type": "string",
            "description": (
                "The type of superhero party "
                "(e.g., 'classic heroes', 'villain masquerade', "
                "'futuristic Gotham')."
            ),
        }
    }

    output_type = "string"

    def forward(self, category: str):
        themes = {
            "classic heroes": (
                "Justice League Gala: Guests come dressed as their "
                "favorite DC heroes with themed cocktails."
            ),
            "villain masquerade": (
                "Gotham Rogues' Ball: A mysterious masquerade where "
                "guests dress as classic Batman villains."
            ),
            "futuristic gotham": (
                "Neo-Gotham Night: A cyberpunk-style party inspired "
                "by Batman Beyond."
            )
        }

        return themes.get(
            category.lower(),
            "Themed party idea not found. Try 'classic heroes', "
            "'villain masquerade', or 'futuristic Gotham'."
        )

## 5. Creating the Multi-Tool Agent

Now I combine the built-in tools and custom tools into a single `CodeAgent`.

The language model acts as the reasoning engine, while the tools provide additional capabilities such as web search, webpage access, menu suggestions, catering recommendations, and superhero-themed party ideas.

The agent can decide which available tool is useful for completing a task.

In [5]:
agent = CodeAgent(
    tools=[
        DuckDuckGoSearchTool(),
        VisitWebpageTool(),
        suggest_menu,
        catering_service_tool,
        SuperheroPartyThemeTool(),
        FinalAnswerTool()
    ],
    model=model,
    max_steps=10,
    verbosity_level=2
)

## 6. Running the Agent

Now I give the multi-tool agent a party planning task.

The agent can analyze the request and decide which of its available tools are useful for completing the task.

In [6]:
agent.run(
    """
    Give me the best playlist for a party at Wayne's mansion.
    The party idea is a 'villain masquerade' theme.
    """
)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Give me the best playlist for a party at Wayne's mansion.                                                       │
│     The party idea is a 'villain masquerade' theme.                                                             │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: To create a villain masquerade-themed playlist for a party at Wayne's mansion, I should first suggest a   
menu idea for such an occasion and then use the search tools to find appropriate playlists.                        
                                                                                                                   
<code>                                                                                                             
menu_suggestion = suggest_menu(occasion="villain masquerade party")                                                
print(menu_suggestion)                                                                                             
                                                                                                                   
playlist_query = "villain masquerade party playlist"                                                               
playlist = web_search(query=playlist_query)                                                                        
print(playlist)                                                                                                    
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  menu_suggestion = suggest_menu(occasion="villain masquerade party")                                              
  print(menu_suggestion)                                                                                           
                                                                                                                   
  playlist_query = "villain masquerade party playlist"                                                             
  playlist = web_search(query=playlist_query)                                                                      
  print(playlist)                                                                                                  
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Custom menu for the butler.
## Search Results

[open.spotify.com › playlist › 4zxYvQs9gByjRhAn5ujSfQyou're at a masquerade and unknowingly dancing with the 
villain](https://open.spotify.com/playlist/4zxYvQs9gByjRhAn5ujSfQ)
you're at a masquerade and unknowingly dancing with the villain · Playlist · 89 songs · 10 likes

[music.youtube.com › playlistMasquerade ball playlist - YouTube 
Music](https://music.youtube.com/playlist?list=PLY6joegkeG1R75YMvOwO6CjVJ1EZQDUJF)
Masquerade ball playlist Erin Antel Playlist • 2026 1.3K views • 39 tracks • 7+ hours Music for any masquerade 
event Sort

[music.youtube.com › playlistMasquerade 
Music](https://music.youtube.com/playlist?list=PLBvN2qT3SK1HbPUX5G13aac2zig6JMrCA)
Masquerade Music Angie Playlist • 2025 52 views • 23 tracks • 1 hour, 47 minutes Sort Fairy Dance

[soundcloud.com › cg5-beats › setsStream CG5 | Listen to Masquerade Party playlist online for 
free...](https://soundcloud.com/cg5-beats/sets/masquerade-party-4)
Listen to Masquerade Party, a playlist curated by CG5 on desktop and mobile.

[soundcloud.com › cg5-beats › masquerade-partyListen to playlists featuring Masquerade Party by CG5 ... 
-...](https://soundcloud.com/cg5-beats/masquerade-party/sets)
Listen to discovery playlists featuring Masquerade Party by CG5 on desktop and mobile.

[dancing with the villain during a masquerade ball - playlist by mkisay | 
Spotify](https://open.spotify.com/playlist/17SEJ9K7DyEAhgoJgZlyNN)
Playlist · dancing with the villain during a masquerade ball · 42 items · 32 saves

[ballroom dancing with the villain - playlist by AsteroidsAway | 
Spotify](https://open.spotify.com/playlist/7D0Z24DSCKpgjTc8sw1PkJ)
Playlist · ballroom dancing with the villain · 30 items · 8 saves

[Villain Mode | Spotify Playlist](https://open.spotify.com/playlist/37i9dQZF1DX3R7OWWGN4gH)
Playlist · Villain Mode · 100 items · 1.4M saves

[The Complete playlist for your next Masquerade party | VIVO 
Masks](https://vivomasks.com/blogs/masquerade/the-complete-playlist-for-your-next-masquerade-party)
May 17, 2020 - But to truly turn back time when masquerades were used to host nights of intrigue and fantasy with a
bit of debauchery on the side, then those are not the tracks you’ll want playing to help you the perfect 
atmosphere. So what would be suitable? Well, we’ve got just what you need create a complete playlist. The 
Masquerade Waltz by Armenian composer Aram Khachaturian is a must-add on your masquerade party playlist.

[Badass Villain Vibes! - playlist by CloudKid | Spotify](https://open.spotify.com/playlist/5HhOezXefwrhf3oeXRTZoj)
Badass Villain Vibes!

Out: None

...

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: Given that the `visit_webpage` tool continues to fail due to the missing required packages, I'll rely     
solely on the `web_search` tool to gather a list of songs for the villain masquerade playlist. I'll manually create
a list of top songs that fit the theme.                                                                            
                                                                                                                   
<code>                                                                                                             
top_songs = [                                                                                                      
    "Suck It Up : Monaleo",                                                                                        
    "No Justice No Peace (Shigaraki Gigantomachia My Hero Academia)",                                              
    "Humanity (Kokushibo Demon Slayer Phonk)",                                                                     
    "Onigashima (feat. PE$O PETE & Breeton Boi)",                                                                  
    "Break Myself! - Eminem",                                                                                      
    "I Will Survive - Gloria Gaynor",                                                                              
    "Thriller - Michael Jackson",                                                                                  
    "Poker Face - Lady Gaga",                                                                                      
    "Bad Guy - Billie Eilish",                                                                                     
    "Smells Like Teen Spirit - Nirvana",                                                                           
    "The Final Countdown - Europe",                                                                                
    "Sweet Child O' Mine - Guns N' Roses",                                                                         
    "Bohemian Rhapsody - Queen",                                                                                   
    "Hotel California - Eagles",                                                                                   
    "Like a Prayer - Madonna",                                                                                     
    "The Devil Went Down to Georgia - Charlie Daniels Band",                                                       
    "Rock with You - Michael Jackson",                                                                             
   [38;2;230;237

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  top_songs = [                                                                                                    
      "Suck It Up : Monaleo",                                                                                      
      "No Justice No Peace (Shigaraki Gigantomachia My Hero Academia)",                                            
      "Humanity (Kokushibo Demon Slayer Phonk)",                                                                   
      "Onigashima (feat. PE$O PETE & Breeton Boi)",                                                                
      "Break Myself! - Eminem",                                                                                    
      "I Will Survive - Gloria Gaynor",                                                                            
      "Thriller - Michael Jackson",                                                                                
      "Poker Face - Lady Gaga",                                                                                    
      "Bad Guy - Billie Eilish",                                                                                   
      "Smells Like Teen Spirit - Nirvana",                                                                         
      "The Final Countdown - Europe",                                                                              
      "Sweet Child O' Mine - Guns N' Roses",                                                                       
      "Bohemian Rhapsody - Queen",                                                                                 
      "Hotel California - Eagles",                                                                                 
      "Like a Prayer - Madonna",                                                                                   
      "The Devil Went Down to Georgia - Charlie Daniels Band",                                                     
      "Rock with You - Michael Jackson",                                                                           
      "The Power - Kanye West, Jay-Z",                                                                             
      "Villein - Zayn Malik",                                                                                      
      "Enemy - Imagine Dragons"                                                                                    
  ]                                                                                                                
                                                                                                                   
  # Creating a playlist description from the top songs                                                             
  playlist_description = ", ".join(top_songs)                                                                      
  final_answer(playlist_description)                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Suck It Up : Monaleo, No Justice No Peace (Shigaraki Gigantomachia My Hero Academia), Humanity 
(Kokushibo Demon Slayer Phonk), Onigashima (feat. PE$O PETE & Breeton Boi), Break Myself! - Eminem, I Will Survive 
- Gloria Gaynor, Thriller - Michael Jackson, Poker Face - Lady Gaga, Bad Guy - Billie Eilish, Smells Like Teen 
Spirit - Nirvana, The Final Countdown - Europe, Sweet Child O' Mine - Guns N' Roses, Bohemian Rhapsody - Queen, 
Hotel California - Eagles, Like a Prayer - Madonna, The Devil Went Down to Georgia - Charlie Daniels Band, Rock 
with You - Michael Jackson, The Power - Kanye West, Jay-Z, Villein - Zayn Malik, Enemy - Imagine Dragons

[Step 6: Duration 7.86 seconds| Input tokens: 26,390 | Output tokens: 969]

"Suck It Up : Monaleo, No Justice No Peace (Shigaraki Gigantomachia My Hero Academia), Humanity (Kokushibo Demon Slayer Phonk), Onigashima (feat. PE$O PETE & Breeton Boi), Break Myself! - Eminem, I Will Survive - Gloria Gaynor, Thriller - Michael Jackson, Poker Face - Lady Gaga, Bad Guy - Billie Eilish, Smells Like Teen Spirit - Nirvana, The Final Countdown - Europe, Sweet Child O' Mine - Guns N' Roses, Bohemian Rhapsody - Queen, Hotel California - Eagles, Like a Prayer - Madonna, The Devil Went Down to Georgia - Charlie Daniels Band, Rock with You - Michael Jackson, The Power - Kanye West, Jay-Z, Villein - Zayn Malik, Enemy - Imagine Dragons"

## 7. What I Learned

In this notebook, I built a multi-tool AI agent using the Hugging Face `smolagents` library.

- A `CodeAgent` can be provided with multiple tools.
- Built-in tools can provide capabilities such as web search and webpage access.
- Custom tools can be created using the `@tool` decorator.
- Custom tools can also be created by inheriting from the `Tool` class.
- The language model acts as the reasoning engine of the agent.
- The agent can decide which available tools are useful for completing a task.
- `max_steps` limits the number of steps the agent can take.
- `verbosity_level=2` provides detailed information about the agent's execution.